# HW3 — Two retrievers and a model: what RAG actually sends

**Name:Ivaikina Darya**
**Student ID: S23070567**
**Group:4**
**Repository:https://github.com/nastoikazhenshen/css4007-hw3-ivaikina**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

>I used ChatGPT to understand the assignment requirements, explain BM25, stemming, embeddings, cosine similarity, and Retrieval-Augmented Generation (RAG).

>I also used ChatGPT to assist in drafting Python code, checking mathematical calculations, and structuring the experimental analysis.

>The retrieval results, model responses, and evaluation metrics are obtained by running the code in this notebook and are checked against the provided data.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9|
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer|10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6|

**avgdl = (9 + 10 + 6) / 3 = 25 / 3 = 8.333333**

**Query terms: how, does, rag, retrieve, information**

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.945910 |
| does | 0 | 0 | 0 | 0 | 1.945910 |
| rag | 0 | 0 | 0 | 0 | 1.945910 |
| retrieve | 0 | 0 | 0 | 0 | 1.945910 |
| information | 0 | 0 | 0 | 0 | 1.945910 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

>BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

>BM25(D2) = 0 + 0 + 0 + 0 + 0 = 0

>BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0 | Tied |
| D2 | 0 | Tied |
| D3 | 0 | Tied |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl = avgdl = (9 + 10 + 6) / 3 = 8.333333**

**Stemmed query terms:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.945910 |
| doe | 0 | 0 | 0 | 0 | 1.945910 |
| rag | 0 | 0 | 0 | 0 | 1.945910 |
| retriev | 0 | 2 | 0 | 1 | 0.510826 |
| inform | 0 | 0 | 0 | 0 | 1.945910 |

**BM25 of each document**, worked:

>BM25(D2) = 0.510826 * 5 / 3.725

>BM25(D2) = 0.685672

>BM25(D3) = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0 | Tied second |
| D2 | 0.685672 | 1 |
| D3 | 0 | Tied second |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

>Without stemming, all documents have a score of zero, so none is uniquely ranked first.
With stemming, D2 becomes the top-ranked document because "retrieval" and "retrieves" are converted to "retriev", matching the stem of "retrieve" from the query.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---:|---:|---:|---:|
| Query - D1 | 0.78 | 0.964365 | 0.812404 | 0.995591 |
| Query - D2 | 1.08 | 0.964365 | 1.122497 | 0.997693 |
| Query - D3 | 0.34 | 0.964365 | 0.830662 | 0.424437 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.995591 | 2 |
| D2 | 0.997693 | 1 |
| D3 | 0.424437 | 3 |

**Which document is most similar to the query?**

>D2 is the most similar document, with cosine similarity approximately 0.997693.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

>Without stemming, BM25 gives all documents a score of zero, so there is no unique ranking.
With stemming, BM25 ranks D2 first, while D1 and D3 are tied. Cosine similarity ranks D2 first, D1 second and D3 third.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

>BM25 relies on matching tokens. The query contains "retrieve", but D2 contains "retrieval" and "retrieves".The supplied embedding vectors allow cosine similarity to identify D2 as the closest document, even though lexical matching originally failed.


Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [ ]:
import sys, os, subprocess, importlib.util
from pathlib import Path
required = {"numpy":"numpy>=1.24,<2.3", "rank_bm25":"rank-bm25>=0.2.2", "sentence_transformers":"sentence-transformers>=5.1", "sklearn":"scikit-learn>=1.3", "tiktoken":"tiktoken>=0.7", "openai":"openai>=1.40", "dotenv":"python-dotenv>=1.0"}
missing = [package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
import urllib.request, json
project_dir = Path('/content/css4007-hw3-ivaikina') if Path('/content').is_dir() else Path.cwd()
(project_dir / 'data').mkdir(parents=True, exist_ok=True)
base='https://raw.githubusercontent.com/nastoikazhenshen/css4007-hw3-ivaikina/main/data/'
for name in ['corpus.jsonl','queries.json','qa.json']:
    file = project_dir / 'data' / name
    if not file.exists():
        urllib.request.urlretrieve(base+name,str(file))
os.chdir(project_dir)
print('Working directory:',Path.cwd())
print('Data files:',*[str((project_dir/'data'/name).exists()) for name in ['corpus.jsonl','queries.json','qa.json']])

Working directory: /content/css4007-hw3-ivaikina
Data files: True True True


In [ ]:

import math
import numpy as np

query = np.array([0.8, 0.2, 0.5])

documents = {
    "D1": np.array([0.7, 0.1, 0.4]),
    "D2": np.array([0.9, 0.3, 0.6]),
    "D3": np.array([0.1, 0.8, 0.2])
}

print("COSINE SIMILARITY:")

for name, doc in documents.items():
    dot = np.dot(query, doc)
    q_norm = np.linalg.norm(query)
    d_norm = np.linalg.norm(doc)
    cosine = dot / (q_norm * d_norm)

    print(
        name,
        "dot =", round(dot, 6),
        "query_norm =", round(q_norm, 6),
        "document_norm =", round(d_norm, 6),
        "cosine =", round(cosine, 6)
    )

N = 3
df = 1
tf = 2
k1 = 1.5
b = 0.75
avgdl = 25 / 3
doc_length = 10

idf = math.log((N - df + 0.5) / (df + 0.5))

bm25_d2 = idf * (
    tf * (k1 + 1)
) / (
    tf + k1 * (1 - b + b * doc_length / avgdl)
)

print("\nBM25 with stemming:")
print("D1 =", 0)
print("D2 =", round(bm25_d2, 6))
print("D3 =", 0)


COSINE SIMILARITY:
D1 dot = 0.78 query_norm = 0.964365 document_norm = 0.812404 cosine = 0.995591
D2 dot = 1.08 query_norm = 0.964365 document_norm = 1.122497 cosine = 0.997693
D3 dot = 0.34 query_norm = 0.964365 document_norm = 0.830662 cosine = 0.424437

BM25 with stemming:
D1 = 0
D2 = 0.685672
D3 = 0


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [ ]:
import json, re, os, numpy as np
from pathlib import Path
with open(Path('data/corpus.jsonl'),encoding='utf-8') as f:
    corpus=[json.loads(line) for line in f if line.strip()]
with open(Path('data/queries.json'),encoding='utf-8') as f:
    fixed_queries=json.load(f)['queries']
with open(Path('data/qa.json'),encoding='utf-8') as f:
    qa_questions=json.load(f)['questions']
texts=[doc['text'] for doc in corpus]
assert (len(corpus),len(fixed_queries),len(qa_questions))==(74,8,6)
print('Chunks:',len(corpus),'Queries:',len(fixed_queries),'QA questions:',len(qa_questions))

Chunks: 74 Queries: 8 QA questions: 6


In [ ]:

from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(
        r"[^\W_]+(?:-[^\W_]+)*",
        text.casefold(),
        flags=re.UNICODE
    )

tokenized_corpus = [
    tokenize(text)
    for text in texts
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

def retrieve_bm25(query, k=3):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(
        -scores,
        kind="stable"
    )[:k]

    return [
        (int(i), float(scores[i]))
        for i in top_indices
    ]

test_query = "What is form GR-305 used for?"

results = retrieve_bm25(test_query, k=3)

print("BM25 test:")
for rank, (index, score) in enumerate(results, 1):
    print(
        rank,
        corpus[index]["id"],
        round(score, 5)
    )


BM25 test:
1 GO-08 13.30649
2 GO-12 6.45334
3 GO-06 5.28268


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [ ]:
import os, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from google.colab import userdata
    hf_token=userdata.get('HF_TOKEN')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    hf_token=os.getenv('HF_TOKEN')
if not hf_token:
    raise ValueError('Provide HF_TOKEN in Colab Secrets (Notebook access ON) or in local .env')
embedding_model=SentenceTransformer('google/embeddinggemma-300m',token=hf_token,device='cpu')
document_embeddings=embedding_model.encode_document(texts,batch_size=2,show_progress_bar=True,convert_to_numpy=True)
assert document_embeddings.shape[0]==74
print('Embedding matrix:',document_embeddings.shape)
def retrieve_embedding(query,k=3):
    query_embedding=embedding_model.encode_query([query],convert_to_numpy=True)
    scores=cosine_similarity(query_embedding,document_embeddings)[0]
    indices=np.argsort(-scores,kind='stable')[:k]
    return [(int(i),float(scores[i])) for i in indices]
for rank,(index,score) in enumerate(retrieve_embedding('What is form GR-305 used for?'),1):
    print(rank,corpus[index]['id'],f'{score:.6f}')

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Embedding matrix: (74, 768)
1 GO-08 0.695517
2 GO-12 0.460873
3 GO-23 0.450486


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | GR-210 missing signature returned unprocessed | GO-07 | BM25. The query contains the exact form identifier GR-210 and words directly appearing in GO-07. |
| Q-10 | Через сколько дней нужно сообщить об изменении дохода семьи во время учебного года? | GO-13 | EmbeddingGemma. The query is in Russian, but the relevant document is written in English. |

In [ ]:

import json
from pathlib import Path

data_dir = Path("/content/css4007-hw3-ivaikina/data")

with open(data_dir / "queries.json", encoding="utf-8") as f:
    fixed_queries = json.load(f)["queries"]

own_queries = [
    {
        "id": "Q-09",
        "text": "GR-210 missing signature returned unprocessed",
        "relevant": ["GO-07"]
    },
    {
        "id": "Q-10",
        "text": "Через сколько дней нужно сообщить об изменении дохода семьи во время учебного года?",
        "relevant": ["GO-13"]
    }
]

all_queries = fixed_queries + own_queries

print("Fixed queries:", len(fixed_queries))
print("Own queries:", len(own_queries))
print("Total queries:", len(all_queries))

for item in all_queries:
    print(item["id"], "-", item["text"])

assert len(all_queries) == 10


Fixed queries: 8
Own queries: 2
Total queries: 10
Q-01 - What is form GR-305 used for?
Q-02 - My family is poor. Is there financial help for students like me?
Q-03 - Өтінішті қашанға дейін тапсыру керек?
Q-04 - What is the minimum GPA for the grant?
Q-05 - When was Astana renamed Nur-Sultan, and why?
Q-06 - Where did the Soviet Union test its nuclear weapons?
Q-07 - Can the grant be sent to my mother's card?
Q-08 - If my grades drop in the autumn, do I still get paid in February?
Q-09 - GR-210 missing signature returned unprocessed
Q-10 - Через сколько дней нужно сообщить об изменении дохода семьи во время учебного года?


## Results for all ten or more queries: top 3 from each retriever

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi
from IPython.display import display, Markdown

def tokenize_bm25(text):
    return re.findall(
        r"[^\W_]+(?:-[^\W_]+)*",
        text.casefold(),
        flags=re.UNICODE
    )

tokenized_corpus = [
    tokenize_bm25(doc["text"])
    for doc in corpus
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

def retrieve_bm25(query, k=3):
    scores = bm25.get_scores(tokenize_bm25(query))
    top_indices = np.argsort(-scores, kind="stable")[:k]

    return [
        (int(i), float(scores[i]))
        for i in top_indices
    ]

comparison_results = []

for item in all_queries:
    query_id = item["id"]
    query_text = item["text"]
    relevant = set(item["relevant"])

    bm25_results = retrieve_bm25(query_text, k=3)
    embedding_results = retrieve_embedding(query_text, k=3)

    print("\n" + "=" * 70)
    print("QUERY:", query_id)
    print("QUESTION:", query_text)
    print("RELEVANT:", sorted(relevant))

    print("\nBM25 TOP-3:")

    for rank, (index, score) in enumerate(bm25_results, 1):
        doc = corpus[index]
        print(f"{rank}. ID: {doc['id']} | Score: {score:.6f}")
        print("First line:", doc["text"].splitlines()[0])

    print("\nEMBEDDING TOP-3:")

    for rank, (index, score) in enumerate(embedding_results, 1):
        doc = corpus[index]
        print(f"{rank}. ID: {doc['id']} | Score: {score:.6f}")
        print("First line:", doc["text"].splitlines()[0])

    bm25_ids = [
        corpus[index]["id"]
        for index, score in bm25_results
    ]

    embedding_ids = [
        corpus[index]["id"]
        for index, score in embedding_results
    ]

    bm25_hit = int(
        any(doc_id in relevant for doc_id in bm25_ids)
    )

    embedding_hit = int(
        any(doc_id in relevant for doc_id in embedding_ids)
    )

    print("\nBM25 hit@3:", bm25_hit)
    print("Embedding hit@3:", embedding_hit)

    comparison_results.append({
        "id": query_id,
        "bm25_ids": bm25_ids,
        "embedding_ids": embedding_ids,
        "bm25_hit": bm25_hit,
        "embedding_hit": embedding_hit
    })

table_lines = [
    "| Query | BM25 top 3 | BM25 hit@3 | Embedding top 3 | Embedding hit@3 |",
    "|---|---|---:|---|---:|"
]

for row in comparison_results:
    table_lines.append(
        f"| {row['id']} "
        f"| {', '.join(row['bm25_ids'])} "
        f"| {row['bm25_hit']} "
        f"| {', '.join(row['embedding_ids'])} "
        f"| {row['embedding_hit']} |"
    )

bm25_total = sum(r["bm25_hit"] for r in comparison_results)
embedding_total = sum(r["embedding_hit"] for r in comparison_results)
n_queries = len(comparison_results)

table_lines.append(
    f"| **Total** | | **{bm25_total}/{n_queries}** "
    f"| | **{embedding_total}/{n_queries}** |"
)

display(Markdown("\n".join(table_lines)))

print("\nFINAL RESULTS")
print("BM25:", bm25_total, "/", n_queries)
print("EmbeddingGemma:", embedding_total, "/", n_queries)

for label, condition in [
    ("BOTH SUCCEEDED", lambda r: r["bm25_hit"] == 1 and r["embedding_hit"] == 1),
    ("BM25 BETTER", lambda r: r["bm25_hit"] == 1 and r["embedding_hit"] == 0),
    ("EMBEDDINGS BETTER", lambda r: r["bm25_hit"] == 0 and r["embedding_hit"] == 1)
]:
    print("\n" + label)

    for row in comparison_results:
        if condition(row):
            print(
                row["id"],
                "BM25:", row["bm25_ids"],
                "Embedding:", row["embedding_ids"]
            )



QUERY: Q-01
QUESTION: What is form GR-305 used for?
RELEVANT: ['GO-08', 'GO-12', 'GO-23']

BM25 TOP-3:
1. ID: GO-08 | Score: 13.306495
First line: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
2. ID: GO-12 | Score: 6.453340
First line: A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.
3. ID: GO-06 | Score: 5.282675
First line: Form GR-104 is the 

| Query | BM25 top 3 | BM25 hit@3 | Embedding top 3 | Embedding hit@3 |
|---|---|---:|---|---:|
| Q-01 | GO-08, GO-12, GO-06 | 1 | GO-08, GO-12, GO-23 | 1 |
| Q-02 | KZ-17, GO-20, KZ-25 | 0 | GO-01, GO-10, GO-20 | 1 |
| Q-03 | GO-14, KZ-01, KZ-02 | 1 | GO-14, GO-09, GO-11 | 1 |
| Q-04 | GO-05, GO-17, GO-19 | 0 | GO-02, GO-05, GO-17 | 1 |
| Q-05 | KZ-05, KZ-06, KZ-11 | 1 | KZ-05, KZ-06, KZ-03 | 1 |
| Q-06 | KZ-37, KZ-38, KZ-03 | 1 | KZ-37, KZ-38, KZ-10 | 1 |
| Q-07 | GO-21, GO-16, GO-06 | 0 | GO-15, GO-18, GO-01 | 1 |
| Q-08 | GO-17, GO-05, GO-04 | 1 | GO-17, GO-18, GO-15 | 1 |
| Q-09 | GO-07, GO-11, GO-17 | 1 | GO-07, GO-10, GO-13 | 1 |
| Q-10 | GO-23, KZ-01, KZ-02 | 0 | GO-13, GO-15, GO-23 | 1 |
| **Total** | | **6/10** | | **10/10** |


FINAL RESULTS
BM25: 6 / 10
EmbeddingGemma: 10 / 10

BOTH SUCCEEDED
Q-01 BM25: ['GO-08', 'GO-12', 'GO-06'] Embedding: ['GO-08', 'GO-12', 'GO-23']
Q-03 BM25: ['GO-14', 'KZ-01', 'KZ-02'] Embedding: ['GO-14', 'GO-09', 'GO-11']
Q-05 BM25: ['KZ-05', 'KZ-06', 'KZ-11'] Embedding: ['KZ-05', 'KZ-06', 'KZ-03']
Q-06 BM25: ['KZ-37', 'KZ-38', 'KZ-03'] Embedding: ['KZ-37', 'KZ-38', 'KZ-10']
Q-08 BM25: ['GO-17', 'GO-05', 'GO-04'] Embedding: ['GO-17', 'GO-18', 'GO-15']
Q-09 BM25: ['GO-07', 'GO-11', 'GO-17'] Embedding: ['GO-07', 'GO-10', 'GO-13']

BM25 BETTER

EMBEDDINGS BETTER
Q-02 BM25: ['KZ-17', 'GO-20', 'KZ-25'] Embedding: ['GO-01', 'GO-10', 'GO-20']
Q-04 BM25: ['GO-05', 'GO-17', 'GO-19'] Embedding: ['GO-02', 'GO-05', 'GO-17']
Q-07 BM25: ['GO-21', 'GO-16', 'GO-06'] Embedding: ['GO-15', 'GO-18', 'GO-01']
Q-10 BM25: ['GO-23', 'KZ-01', 'KZ-02'] Embedding: ['GO-13', 'GO-15', 'GO-23']


In [ ]:
additional = [
 ('Q-11','Which city was classified as a Gamma+ level global city by the Globalization and World Cities Research Network?',['KZ-07']),
 ('Q-12','ak-serke shorpo kymyz ayran',['KZ-25']),
 ('Q-13','Töretam Trans-Aral Railway 2005 lease',['KZ-10']),
 ('Q-14','Dike Kokaral 2008 42 m 2013 salinity',['KZ-20']),
 ('Q-15','Alpharabius Syriac Aristotelian Neoplatonism',['KZ-29'])]
for qid,question,relevant in additional:
    print('\nDIAGNOSTIC',qid,question,'relevant:',relevant)
    for label,retriever in [('BM25',retrieve_bm25),('Embedding',retrieve_embedding)]:
        result=retriever(question,3)
        print(label,[(corpus[i]['id'],round(score,6)) for i,score in result], 'hit@3:',int(any(corpus[i]['id'] in relevant for i,_ in result)))


DIAGNOSTIC Q-11 Which city was classified as a Gamma+ level global city by the Globalization and World Cities Research Network? relevant: ['KZ-07']
BM25 [('KZ-07', 42.99722), ('KZ-06', 14.827385), ('KZ-08', 14.475751)] hit@3: 1
Embedding [('KZ-07', 0.427459), ('KZ-08', 0.281159), ('KZ-06', 0.265983)] hit@3: 1

DIAGNOSTIC Q-12 ak-serke shorpo kymyz ayran relevant: ['KZ-25']
BM25 [('KZ-25', 18.326559), ('KZ-01', 0.0), ('KZ-02', 0.0)] hit@3: 1
Embedding [('KZ-25', 0.426974), ('KZ-24', 0.321599), ('KZ-48', 0.311835)] hit@3: 1

DIAGNOSTIC Q-13 Töretam Trans-Aral Railway 2005 lease relevant: ['KZ-10']
BM25 [('KZ-10', 14.718061), ('KZ-08', 2.767386), ('KZ-20', 2.497646)] hit@3: 1
Embedding [('KZ-18', 0.301439), ('KZ-20', 0.289599), ('KZ-10', 0.285383)] hit@3: 1

DIAGNOSTIC Q-14 Dike Kokaral 2008 42 m 2013 salinity relevant: ['KZ-20']
BM25 [('KZ-20', 19.18419), ('KZ-46', 6.242739), ('KZ-36', 4.381494)] hit@3: 1
Embedding [('KZ-20', 0.540543), ('KZ-18', 0.320232), ('KZ-19', 0.312257)] hit@3: 1

## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | 1 | GO-08, GO-12, GO-23 | 1 |
| Q-02 | KZ-17, GO-20, KZ-25 | 0 | GO-01, GO-10, GO-20 | 1 |
| Q-03 | GO-14, KZ-01, KZ-02 | 1 | GO-14, GO-09, GO-11 | 1 |
| Q-04 | GO-05, GO-17, GO-19 | 0 | GO-02, GO-05, GO-17 | 1 |
| Q-05 | KZ-05, KZ-06, KZ-11 | 1 | KZ-05, KZ-06, KZ-03 | 1 |
| Q-06 | KZ-37, KZ-38, KZ-03 | 1 | KZ-37, KZ-38, KZ-10 | 1 |
| Q-07 | GO-21, GO-16, GO-06 | 0 | GO-15, GO-18, GO-01 | 1 |
| Q-08 | GO-17, GO-05, GO-04 | 1 | GO-17, GO-18, GO-15 | 1 |
| Q-09 | GO-07, GO-11, GO-17 | 1 | GO-07, GO-10, GO-13 | 1 |
| Q-10 | GO-23, KZ-01, KZ-02 | 0 | GO-13, GO-15, GO-23 | 1 |
| **hit@3 total** | | **6/10** | | **10/10** |

**Both retrievers agree:** which query, and why was it easy?

>For Q-01, both BM25 and EmbeddingGemma achieved hit@3 = 1. BM25 retrieved GO-08, GO-12, and GO-06, while EmbeddingGemma retrieved GO-08, GO-12, and GO-23.
Both ranked GO-08 first. The question contains the exact form identifier "GR-305", which also appears in GO-08 and GO-12. EmbeddingGemma additionally retrieved GO-23, a Russian-language document describing the same appeal procedure.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

>For the additional query Q-13, "Töretam Trans-Aral Railway 2005 lease", the relevant document was KZ-10.
BM25 ranked KZ-10 first with a score of 14.718061, followed by KZ-08 and KZ-20.
EmbeddingGemma ranked KZ-18 first (0.301439), KZ-20 second (0.289599), and the relevant KZ-10 third (0.285383).
BM25 performed better in terms of ranking position because it matched rare, specific strings such as "Töretam", "Trans-Aral Railway", "2005", and "lease". EmbeddingGemma placed documents associated with the Aral Sea above the exact target.


**Embeddings did better:** which query, and what could BM25 not match?

>GO-15 is written in Russian and states that the bank account must belong to the student and transfers to parents or other people are not allowed. EmbeddingGemma successfully matched the English question about a mother's bank card with the Russian rule about transferring money to parents. BM25 failed because the important words in the English query did not match the Russian wording.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [ ]:
from openai import OpenAI
from pathlib import Path
import os
MODEL='gpt-5.6-luna'
USE_LIVE_API=False
RAG_SYSTEM=('Answer the question using only the provided context.\n'
'If the answer cannot be found in the context, say that you do not know.\n'
'Cite the context blocks you used, like [1] or [2][3].')
def format_context_blocks(retrieved):
    return [f"[{rank}]\nChunk ID: {corpus[index]['id']}\nTitle: {corpus[index]['title']}\nText: {corpus[index]['text']}" for rank,(index,_) in enumerate(retrieved,1)]
def build_messages(question,method,k=3):
    if method=='llm_only': return [{'role':'user','content':question}],[],[]
    if method=='bm25': retrieved=retrieve_bm25(question,k)
    elif method=='embedding': retrieved=retrieve_embedding(question,k)
    else: raise ValueError(method)
    blocks=format_context_blocks(retrieved)
    user_prompt='Context:\n'+'\n\n'.join(blocks)+'\n\nQuestion:\n'+question
    return [{'role':'system','content':RAG_SYSTEM},{'role':'user','content':user_prompt}],retrieved,blocks
def get_api_client():
    try:
        from google.colab import userdata
        key=userdata.get('OPENAI_API_KEY')
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv(override=True)
        key=os.getenv('OPENAI_API_KEY')
    if not key: raise ValueError('OPENAI_API_KEY not configured')
    return OpenAI(api_key=key)
def ask_llm(messages):
    response=get_api_client().chat.completions.create(model=MODEL,messages=messages)
    return response.choices[0].message.content or '',response.usage.prompt_tokens
print('Part C ready; USE_LIVE_API =',USE_LIVE_API)

Part C ready; USE_LIVE_API = False


## C1 — the complete prompt, printed before it is sent

In [ ]:
import tiktoken
selected_qa=next(q for q in qa_questions if q['id']=='C-03')
messages,retrieved,blocks=build_messages(selected_qa['question'],'embedding',3)
for msg in messages:
    print('ROLE:',msg['role'].upper(),'\n'+msg['content']+'\n')
enc=tiktoken.get_encoding('o200k_base')
count_tokens=lambda value:len(enc.encode(value))
parts=[count_tokens(messages[0]['content']),count_tokens(selected_qa['question']),count_tokens('\n\n'.join(blocks))]
visible=sum(count_tokens(m['content']) for m in messages)
print('System instructions:',parts[0],'Question:',parts[1],'Retrieved context:',parts[2])
print('Sum of three parts:',sum(parts),'Visible message contents:',visible)
for i,block in enumerate(blocks,1): print('Chunk',i,count_tokens(block),'tokens',len(block),'characters')
print('Context characters:',len('\n\n'.join(blocks)))
if USE_LIVE_API:
    answer,usage=ask_llm(messages)
    print('NEW LIVE API RESULT')
else:
    answer='The minimum GPA needed is **2.67 on a 4.0 scale**. [1]'
    usage=379
    print('SAVED RESULT FROM PREVIOUS C1 RUN (not a new API call)')
print('ANSWER:',answer)
print('usage.prompt_tokens:',usage,'Visible content:',visible,'Difference:',usage-visible)

ROLE: SYSTEM 
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

ROLE: USER 
Context:
[1]
Chunk ID: GO-02
Title: Who qualifies
Text: An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[2]
Chunk ID: GO-20
Title: Who cannot apply
Text: The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which th

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 14 |
| Retrieved context | 307 |
| **Total input (sum of parts)** | **365** |
| Complete visible message contents | 369 |
| **`usage.prompt_tokens` from the response** | **379** |

- Number of retrieved chunks: 3
- Chunk 1 (GO-02): 119 tokens, 499 characters
- Chunk 2 (GO-20): 72 tokens, 366 characters
- Chunk 3 (GO-05): 116 tokens, 414 characters
- Total context size: 307 tokens and 1,283 characters

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>The sum of the three separately counted parts is 365 tokens, while the complete visible message contents contain 369 tokens. This difference occurs because the complete prompt includes additional formatting, such as the "Context:" and "Question:" labels, and tokenization across combined text can differ from tokenizing separate sections.

**What exactly does retrieval add to the model's input?**

>Retrieval adds the actual text of the three selected document chunks to the prompt, including their numbered references, chunk IDs, and titles.

## C2 — three systems, six questions

In [ ]:
from IPython.display import display, Markdown
SAVED_C2 = {'C-01': {'llm_only': ('Which student support scheme or country do you mean? “Income band 2” varies by program, and the amount and payment date depend on the academic year and whether it’s a grant, allowance, or bursary.', 24), 'bm25': ('A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]', 419), 'embedding': ('A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]', 411)}, 'C-02': {'llm_only': ('The appeal deadline and form depend on the grant program, funding body, and country. They should be stated in your refusal/decision letter.', 27), 'bm25': ('You have **14 calendar days** from the date on the decision letter to appeal. Use **Form GR-305**. [3]', 392), 'embedding': ('You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [2]', 374)}, 'C-03': {'llm_only': ('Which need-based grant do you mean? GPA requirements vary by program, and some—such as the federal Pell Grant—don’t have a minimum GPA for initial eligibility, though satisfactory academic progress is required to continue receiving aid.', 20), 'bm25': ('The minimum GPA needed is **2.67 on a 4.0 scale**. [1]', 405), 'embedding': ('The minimum GPA needed is **2.67 on a 4.0 scale**. [1]', 379)}, 'C-04': {'llm_only': ('Kazakhstan formally decided to switch from Cyrillic to a Latin-based Kazakh alphabet on **27 October 2017**, under a decree signed by President Nursultan Nazarbayev. The original target was to complete the transition by **2025**, but the timetable was later extended. It is now scheduled to be completed by **2031**, in stages.', 32), 'bm25': ('Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The completion deadline was initially 2025 but was later pushed back to **2031** in January 2021. [1][2]', 644), 'embedding': ('Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition, initially planned for 2025, is now due to finish in **2031**. [1][2]', 623)}, 'C-05': {'llm_only': ('Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy** (Алма-Атинский институт народного хозяйства).', 23), 'bm25': ('Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**. [1]', 556), 'embedding': ('Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**. [1]', 651)}, 'C-06': {'llm_only': ('The amount depends heavily on your country, university, program, enrollment status, income/assets, and family circumstances. If you mean the United States: Federal Pell Grant is generally not available for master’s students, and other funding options vary. The response requests country, university, study status, income and financial aid details.', 21), 'bm25': ('I don’t know the amount. The need-based grant described here is only for full-time undergraduate students; master’s students are covered by a separate scheme and should ask their department for its rules. [2][3]', 351), 'embedding': ('You cannot apply for this need-based grant because it is only for full-time undergraduate students. Master’s students are covered by a separate scheme and should ask their department about its rules. [1]', 351)}}
c2_results={}
for item in qa_questions:
    qid=item['id']
    c2_results[qid]={}
    print('\nQUESTION',qid,item['question'],'\nREFERENCE:',item['reference'])
    for method in ('llm_only','bm25','embedding'):
        messages,retrieved,blocks=build_messages(item['question'],method,3)
        if USE_LIVE_API:
            answer,tokens=ask_llm(messages)
            source='NEW LIVE API RESPONSE'
        else:
            answer,tokens=SAVED_C2[qid][method]
            source='PREVIOUSLY RECORDED C2 RESPONSE (not a new API call)'
        ids=[corpus[index]['id'] for index,score in retrieved]
        c2_results[qid][method]={'answer':answer,'prompt_tokens':tokens,'retrieved_ids':ids}
        print(method,'|',source,'| retrieved:',ids,'| input tokens:',tokens,'\nANSWER:',answer)
print('\nC2 RESULTS LOADED FOR C3:',len(c2_results),'questions')
print('Note: LLM-only C-06 stored text is an abbreviated record of the earlier response; other records reflect prior answers.')


QUESTION C-01 How much does a student in income band 2 receive, and when is it paid? 
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
llm_only | PREVIOUSLY RECORDED C2 RESPONSE (not a new API call) | retrieved: [] | input tokens: 24 
ANSWER: Which student support scheme or country do you mean? “Income band 2” varies by program, and the amount and payment date depend on the academic year and whether it’s a grant, allowance, or bursary.
bm25 | PREVIOUSLY RECORDED C2 RESPONSE (not a new API call) | retrieved: ['GO-04', 'GO-02', 'GO-01'] | input tokens: 419 
ANSWER: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]
embedding | PREVIOUSLY RECORDED C2 RESPONSE (not a new API call) | retrieved: ['GO-04', 'GO-03', 'GO-15'] | input tokens: 411 
ANSWER: A student in income band 2 receives **150,000 tenge for the academic year*

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Did not provide the grant amount or payment dates | ✓ 150,000 tenge, October and February | ✓ 150,000 tenge, 75,000 in October and February |
| C-02 | ✗ Did not identify the deadline or form | ✓ 14 calendar days, GR-305 | ✓ 14 calendar days, GR-305 |
| C-03 | ✗ Did not provide the minimum GPA | ✓ GPA 2.67, current rule | ✓ GPA 2.67, current rule |
| C-04 | ✓ October 2017, initial target 2025, current target 2031 | ✓ October 2017, revised to 2031 in January 2021 | ✓ October 2017, current target 2031 |
| C-05 | ✓ 1963, Alma-Ata Institute of National Economy | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy |
| C-06 | ✗ Generic information about other grant programs | ✓ Explicitly states the amount is unknown and refers to a separate scheme | ✗ Identifies the separate scheme but does not explicitly state that the amount is unknown |
| **Agrees with reference** | **2/6** | **6/6** | **5/6** |

## C3 — question 1 of 3: C-06

**1. What BM25 retrieved** (top 3, ids and scores):

>GO-16 (13.120353), GO-01 (11.907722), GO-20 (11.153686). Hit@3 = 1.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>GO-20 (0.666420), GO-16 (0.616117), GO-01 (0.553895). Hit@3 = 1.


**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-16, [2] GO-01, [3] GO-20.

>Embedding: [1] GO-20, [2] GO-16, [3] GO-01.

The full context blocks are printed in the code output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM only gave generic financial aid information. BM25 RAG correctly stated that the amount was unknown. Embedding RAG explained the separate scheme but did not explicitly state that the amount was unavailable.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Both retrievers found GO-20, which states that the master's scheme is not described in the handbook. Embedding RAG had a generation failure under strict evaluation: the correct evidence was available, but the answer did not explicitly acknowledge the missing amount. BM25 RAG answered correctly.

In [ ]:
question_id='C-06'
item=next(q for q in qa_questions if q['id']==question_id)
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method in ('bm25','embedding'):
    messages,retrieved,blocks=build_messages(item['question'],method,3)
    print('\n===',method.upper(),'===')
    for rank,(index,score) in enumerate(retrieved,1):
        print(rank,corpus[index]['id'],f'{score:.6f}')
    print('CONTEXT BLOCKS SENT TO MODEL:\n','\n\n'.join(blocks))
    print('RECORDED MODEL ANSWER:\n',c2_results[question_id][method]['answer'])
print('\nRECORDED LLM ONLY ANSWER:\n',c2_results[question_id]['llm_only']['answer'])

QUESTION: I am a master's student. How much need-based grant can I get? 
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

=== BM25 ===
1 GO-16 13.120353
2 GO-01 11.907722
3 GO-20 11.153686
CONTEXT BLOCKS SENT TO MODEL:
 [1]
Chunk ID: GO-16
Title: Combining the grant with other support
Text: The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2]
Chunk ID: GO-01
Title: About the need-based study grant 2026
Text: The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms th

## C3 — question 2 of 3: C-03

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-02 (14.011030), GO-05 (13.779353), GO-11 (12.406370).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-02 (0.628582), GO-20 (0.569065), GO-05 (0.545070).

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-02, [2] GO-05, [3] GO-11.

>Embedding: [1] GO-02, [2] GO-20, [3] GO-05.

The complete context blocks are printed in the code output below.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM only could not identify the minimum GPA. Both RAG systems correctly answered 2.67 and cited GO-02 [1], despite receiving the outdated GPA 3.0 from GO-05.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Neither system failed. Both retrieved GO-02, which states that the cumulative GPA must be at least 2.67. GO-05 contains the archived 2025 requirement of 3.0. Both models correctly distinguished the current rule from the outdated one.

In [15]:
question_id='C-03'
item=next(q for q in qa_questions if q['id']==question_id)
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method in ('bm25','embedding'):
    messages,retrieved,blocks=build_messages(item['question'],method,3)
    print('\n===',method.upper(),'===')
    for rank,(index,score) in enumerate(retrieved,1):
        print(rank,corpus[index]['id'],f'{score:.6f}')
    print('CONTEXT BLOCKS SENT TO MODEL:\n','\n\n'.join(blocks))
    print('RECORDED MODEL ANSWER:\n',c2_results[question_id][method]['answer'])
print('\nRECORDED LLM ONLY ANSWER:\n',c2_results[question_id]['llm_only']['answer'])

QUESTION: What is the minimum GPA needed to qualify for the need-based grant? 
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

=== BM25 ===
1 GO-02 14.011030
2 GO-05 13.779353
3 GO-11 12.406370
CONTEXT BLOCKS SENT TO MODEL:
 [1]
Chunk ID: GO-02
Title: Who qualifies
Text: An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[2]
Chunk ID: GO-05
Title: Archived: the 2025 scheme (no longer in force)
Text: ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band

## C3 — question 3 of 3: C-02

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-05 (17.165174), GO-08 (14.600731), GO-12 (13.572114).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-08 (0.602892), GO-12 (0.559074), GO-11 (0.534273).

**3. What was sent to the model** (the context blocks):

> BM25: [1] GO-05, [2] GO-08, [3] GO-12.

> Embedding: [1] GO-08, [2] GO-12, [3] GO-11.

The full context blocks are printed in the code output.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> LLM only could not identify the appeal deadline or form. Both RAG systems correctly answered 14 calendar days and Form GR-305, citing GO-12.EmbeddingGemma ranked relevant documents higher, while BM25 ranked the irrelevant archived GO-05 first.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Neither system failed to answer correctly. Both retrieved GO-12, which states: "A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305."
Both models used this evidence correctly despite irrelevant additional chunks.

In [16]:
question_id='C-02'
item=next(q for q in qa_questions if q['id']==question_id)
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method in ('bm25','embedding'):
    messages,retrieved,blocks=build_messages(item['question'],method,3)
    print('\n===',method.upper(),'===')
    for rank,(index,score) in enumerate(retrieved,1):
        print(rank,corpus[index]['id'],f'{score:.6f}')
    print('CONTEXT BLOCKS SENT TO MODEL:\n','\n\n'.join(blocks))
    print('RECORDED MODEL ANSWER:\n',c2_results[question_id][method]['answer'])
print('\nRECORDED LLM ONLY ANSWER:\n',c2_results[question_id]['llm_only']['answer'])

QUESTION: My grant application was refused. How long do I have to appeal, and which form do I use? 
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

=== BM25 ===
1 GO-05 17.165174
2 GO-08 14.600731
3 GO-12 13.572114
CONTEXT BLOCKS SENT TO MODEL:
 [1]
Chunk ID: GO-05
Title: Archived: the 2025 scheme (no longer in force)
Text: ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,000 tenge, and the whole amount was paid in a single instalment in November. These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year.

[2]
Chunk ID: GO-08
Title: Form GR-305: appeals
Text: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision th

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores lexical overlap and document statistics; embeddings compare semantic vector similarity. The two approaches can retrieve different documents for the same question.

**2. When can BM25 perform better than embeddings?**

> BM25 can rank exact rare names, numeric strings and identifiers especially well. In the diagnostic Q-13, BM25 placed KZ-10 first while EmbeddingGemma placed it third (both were still hit@3=1).

**3. When can embeddings perform better than BM25?**

> Embeddings can handle paraphrasing and cross-language questions. In Q-07 the English question about a mother's card matched GO-15 in Russian; Embedding hit@3=1 and BM25 hit@3=0.

**4. Why is cosine similarity used with embeddings?**

> It measures directional similarity between two vectors, reducing dependence on vector magnitude. The score helps rank semantically related texts.

**5. What information is actually sent to an LLM in a RAG system?**

> The model receives system instructions, the user's question, and the exact text of retrieved context blocks, including their IDs and titles. In C1 the three context blocks had 307 `o200k_base` tokens.

**6. Does the LLM search the original documents itself?**

> No. Our Python retriever reads the 74 chunks and selects Top-K before sending messages to the LLM. The LLM receives only the selected text, not direct access to the corpus.

**7. What happens if retrieval returns an irrelevant chunk?**

> It can waste input tokens, distract the model, or omit necessary information if relevant chunks fall outside Top-K. Incorrect answers can then result from missing evidence or misuse of available evidence.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?**

> Higher K may include missing relevant documents but also introduces distractions and outdated rules. The next executable cell reports the contexts for K=1,3,8; re-running live LLM responses requires setting `RUN_TOPK_API = True` in that cell. The impact on answer quality must be described from actual outputs rather than guessed.

In [17]:
item=next(q for q in qa_questions if q['id']=='C-03')
RUN_TOPK_API=False
for k in (1,3,8):
    messages,retrieved,blocks=build_messages(item['question'],'embedding',k)
    print('K =',k,'IDs:',[corpus[i]['id'] for i,_ in retrieved])
    if RUN_TOPK_API:
        answer,tokens=ask_llm(messages)
        print('LIVE ANSWER:',answer,'API input tokens:',tokens)
    else:
        print('No new API request; set RUN_TOPK_API=True to measure actual answer changes.')

K = 1 IDs: ['GO-02']
No new API request; set RUN_TOPK_API=True to measure actual answer changes.
K = 3 IDs: ['GO-02', 'GO-20', 'GO-05']
No new API request; set RUN_TOPK_API=True to measure actual answer changes.
K = 8 IDs: ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
No new API request; set RUN_TOPK_API=True to measure actual answer changes.


**9. Is a vector database required to build a RAG system?**

> No. This notebook keeps 74 document embeddings in a NumPy array and computes cosine similarity with scikit-learn. Vector databases can improve scalability but are not necessary for this experiment.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?**

> On the first ten Part B queries, EmbeddingGemma achieved hit@3 = **10/10** versus **6/10** for BM25. Under the strict grading used in the saved C2 assessment, BM25 RAG agreed with reference in **6/6**, while Embedding RAG agreed in **5/6**. C-06 was counted incorrect for Embedding RAG because it omitted an explicit statement that the amount is unknown, even though it identified the separate scheme. The evaluation is therefore sensitive to how completeness is judged.

---
# Conclusion

> The experiment highlights that accurate retrieval and complete generation are different goals. I would next build a hybrid BM25–embedding retriever with reranking, checks for outdated rules, and an explicit evidence-sufficiency check before answering. The remaining Top-K answer-quality experiment should be completed with real model calls if required for final grading.